# RAG Assistant for Chemistry Literature
### AI4Chemistry Bootcamp — Retrieval-Augmented Generation Tutorial

In this notebook you'll build a small retrieval-augmented generation (RAG) system over a corpus of chemistry paper abstracts, then evaluate how well it answers chemistry questions compared to an LLM with no retrieval at all.

**You will learn to:**
- Chunk and embed a corpus of chemistry paper abstracts using a sentence-transformer model
- Build a FAISS vector index and retrieve the top-k most relevant chunks for a query
- Construct a RAG prompt that provides retrieved context to an LLM before answering
- Evaluate answer faithfulness and relevance using automated metrics (BERTScore, embedding-based relevance)
- **(Challenge)** Improve retrieval quality with cross-encoder re-ranking

Cells marked `### YOUR CODE HERE ###` are exercises — fill them in. Discussion questions are marked with 🗣️ — answer them in the markdown cell provided. Some have no single right answer, so compare notes with a neighbour.

## A note on the dataset

This notebook ships with a **40-abstract demo corpus** (`DEMO_CORPUS`, defined below) covering ten chemistry subfields — including several abstracts specifically about **Suzuki-Miyaura coupling solvents**, and one deliberately fictional paper (the "Kessler-Onyema coupling") used later to test hallucination. It's small on purpose, so the whole notebook runs in a couple of minutes on a laptop or a plain Colab CPU runtime, with no external hosting or internet-dependent data download required.

For the real assignment corpus — **500 chemistry paper abstracts** — run the companion script `data_prep_pubmed.py` (provided alongside this notebook) once, ahead of time, with internet access. It queries PubMed's free Entrez API across the same ten topics and saves `chemistry_abstracts.csv`. Point `FULL_CORPUS_PATH` below at that file (or a raw GitHub URL to it, once you've committed it to the course repo) to switch from the demo corpus to the full one — no other code changes needed.

## Part 0 — Setup

In [ ]:
!pip -q install sentence-transformers faiss-cpu openai bert-score pandas matplotlib transformers accelerate

In [ ]:
import json
import os
import time
import getpass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss

print("Imports OK")

In [ ]:
DEMO_CORPUS = [
    {
        "id": "C001",
        "title": "Aqueous-Compatible Suzuki-Miyaura Couplings Using Pd/XPhos Precatalysts",
        "authors": "Reyes, T. et al.",
        "year": 2022,
        "journal": "J. Synth. Methods",
        "topic": "suzuki_coupling",
        "abstract": "We report a broadly applicable protocol for Suzuki-Miyaura cross-coupling of aryl chlorides with boronic acids using a Pd/XPhos precatalyst system. Reactions proceed efficiently in dioxane/water (4:1) at 60-80 C with K2CO3 as base, reaching full conversion within 2 hours for electron-neutral and electron-rich substrates. Electron-poor aryl chlorides required switching to a THF/water mixture with Cs2CO3 to suppress protodeboronation. Ethanol/water was also evaluated and gave comparable yields for sterically unhindered couplings, offering a greener alternative to dioxane. The method tolerates free amines, esters, and unprotected phenols, and was demonstrated on 24 substrates with isolated yields of 74-96%."
    },
    {
        "id": "C002",
        "title": "Solvent Effects on Transmetalation Rates in Suzuki-Type Cross-Coupling",
        "authors": "Okonkwo, L. et al.",
        "year": 2021,
        "journal": "Adv. Catal. Lett.",
        "topic": "suzuki_coupling",
        "abstract": "The rate of transmetalation in Pd-catalyzed Suzuki coupling was studied across a panel of solvents including THF, 1,4-dioxane, DMF, toluene, and ethanol, each with an aqueous co-solvent. Kinetic profiling showed that polar aprotic solvents (DMF, dioxane) accelerate boronate formation but can promote protodeboronation of heteroaryl boronic acids at elevated temperature. Toluene/water biphasic systems gave the slowest but most selective transmetalation, useful for sensitive substrates. Ethanol/water mixtures balanced rate and selectivity for most benchmark couplings. The study provides a practical solvent-selection guide for chemists optimizing Suzuki couplings on unfamiliar substrates."
    },
    {
        "id": "C003",
        "title": "Ligand-Free Suzuki Coupling in Ethanol-Water Mixtures",
        "authors": "Park, J. et al.",
        "year": 2020,
        "journal": "Green Chem. Reports",
        "topic": "suzuki_coupling",
        "abstract": "A ligand-free Pd(OAc)2-catalyzed Suzuki coupling protocol is described that operates in ethanol/water (3:1) at room temperature. The absence of phosphine ligands simplifies purification and reduces cost. Aryl bromides and iodides couple with arylboronic acids in yields of 65-92% using K3PO4 as base. Aryl chlorides remained largely unreactive under these mild, ligand-free conditions, indicating that oxidative addition, not solvent choice, is rate-limiting for less activated electrophiles. This work highlights ethanol/water as a sustainable solvent system for Suzuki couplings of activated aryl halides."
    },
    {
        "id": "C004",
        "title": "Continuous-Flow Suzuki-Miyaura Coupling for Pharmaceutical Intermediates",
        "authors": "Haddad, R. et al.",
        "year": 2023,
        "journal": "J. Synth. Methods",
        "topic": "suzuki_coupling",
        "abstract": "A continuous-flow reactor was developed for Suzuki-Miyaura coupling of pharmaceutically relevant heteroaryl fragments. THF/water was selected as the flow-compatible solvent system after screening dioxane, DMF, and acetonitrile mixtures, based on solubility of both the palladium catalyst and inorganic base. Residence times of 4-6 minutes at 100 C gave conversions exceeding 90% for pyridyl and pyrimidyl boronates. The flow process reduced catalyst loading from 3 mol% (batch) to 0.5 mol% while maintaining yield, illustrating how solvent and reactor engineering jointly improve process efficiency for cross-coupling steps in drug synthesis."
    },
    {
        "id": "C005",
        "title": "Buchwald-Hartwig Amination versus Suzuki Coupling: A Solvent Compatibility Study",
        "authors": "Nilsson, E. et al.",
        "year": 2019,
        "journal": "Adv. Catal. Lett.",
        "topic": "suzuki_coupling",
        "abstract": "This comparative study examines solvent requirements for Buchwald-Hartwig amination and Suzuki-Miyaura coupling when performed sequentially on the same substrate without intermediate purification. Toluene proved compatible with both transformations, while dioxane/water mixtures used for Suzuki coupling required careful drying before the subsequent amination step. A one-pot toluene-based protocol combining both couplings is described, avoiding solvent switches and improving overall yield of a biaryl-amine building block from 58% (stepwise, two solvents) to 79% (one-pot, toluene only)."
    },
    {
        "id": "C006",
        "title": "Microwave-Assisted Suzuki Coupling in Dimethylformamide",
        "authors": "Abara, C. et al.",
        "year": 2018,
        "journal": "J. Synth. Methods",
        "topic": "suzuki_coupling",
        "abstract": "Microwave irradiation was applied to accelerate Suzuki-Miyaura couplings in DMF/water (5:1), reducing reaction times from hours to under 10 minutes for a range of aryl bromides. DMF was chosen over dioxane for its higher microwave absorptivity, giving faster and more uniform heating. Yields were generally comparable to conventional thermal heating in dioxane/water, though a subset of electron-poor boronic acids showed increased protodeboronation under microwave conditions in DMF, an effect attributed to localized superheating rather than bulk solvent temperature."
    },
    {
        "id": "C007",
        "title": "Visible-Light Photoredox Catalysis for C-C Bond Formation",
        "authors": "Weiss, D. et al.",
        "year": 2021,
        "journal": "J. Synth. Methods",
        "topic": "photoredox",
        "abstract": "An Ru(bpy)3-catalyzed photoredox protocol enables C-C bond formation between alpha-amino radicals and Michael acceptors under visible-light irradiation. Reactions proceed at room temperature in degassed acetonitrile, generating quaternary stereocenters with moderate diastereoselectivity. The method complements two-electron enolate chemistry by accessing radical intermediates that are difficult to generate under classical basic conditions, and was applied to the late-stage functionalization of three drug-like scaffolds."
    },
    {
        "id": "C008",
        "title": "Dual Photoredox/Nickel Catalysis for Aryl-Alkyl Cross-Coupling",
        "authors": "Fujimoto, K. et al.",
        "year": 2022,
        "journal": "Adv. Catal. Lett.",
        "topic": "photoredox",
        "abstract": "Merging photoredox catalysis with nickel catalysis enables direct cross-coupling of aryl halides with unactivated alkyl radicals generated from carboxylic acid precursors via decarboxylation. The dual catalytic cycle operates under blue LED irradiation in DMSO, avoiding the need for pre-formed organometallic nucleophiles. This strategy expands the scope of cross-coupling beyond boronic acids and organozinc reagents to abundant, bench-stable carboxylic acid feedstocks."
    },
    {
        "id": "C009",
        "title": "Energy Transfer Photocatalysis for [2+2] Cycloadditions",
        "authors": "Costa, M. et al.",
        "year": 2020,
        "journal": "J. Synth. Methods",
        "topic": "photoredox",
        "abstract": "Triplet energy transfer photocatalysis using an iridium photosensitizer promotes intramolecular [2+2] cycloadditions of tethered enones under mild conditions. Unlike direct UV irradiation, the triplet-sensitized pathway avoids competing Norrish-type fragmentation, improving yields of cyclobutane-fused products to 70-88% across a substrate series relevant to terpenoid natural product synthesis."
    },
    {
        "id": "C010",
        "title": "Organophotoredox Catalysts as Sustainable Alternatives to Iridium Complexes",
        "authors": "Sato, Y. et al.",
        "year": 2023,
        "journal": "Green Chem. Reports",
        "topic": "photoredox",
        "abstract": "Metal-free organic photocatalysts based on acridinium salts are evaluated as replacements for iridium and ruthenium photoredox catalysts in radical cation Diels-Alder reactions. The organic catalysts show comparable quantum yields and improved recyclability from reaction mixtures via simple aqueous extraction, addressing cost and metal-contamination concerns for pharmaceutical-relevant photoredox transformations."
    },
    {
        "id": "C011",
        "title": "Mechanochemical Synthesis: Eliminating Solvent Waste in Amide Coupling",
        "authors": "Dubois, A. et al.",
        "year": 2021,
        "journal": "Green Chem. Reports",
        "topic": "green_chemistry",
        "abstract": "Ball-mill mechanochemistry is applied to amide bond formation between carboxylic acids and amines, achieving yields comparable to solution-phase coupling reagents (HATU, DCC) without any solvent. Reaction times of 20-40 minutes of milling produced amides in 80-95% yield across 15 substrate pairs. Life-cycle analysis indicated a 90% reduction in process mass intensity relative to conventional DMF-based coupling protocols."
    },
    {
        "id": "C012",
        "title": "Deep Eutectic Solvents as Reaction Media for Aldol Condensations",
        "authors": "Ivanova, S. et al.",
        "year": 2019,
        "journal": "Green Chem. Reports",
        "topic": "green_chemistry",
        "abstract": "Choline chloride/urea deep eutectic solvent was investigated as a biodegradable, low-toxicity medium for base-catalyzed aldol condensations. The DES both dissolves the reactants and stabilizes the enolate intermediate through hydrogen bonding, giving yields 10-15% higher than the same reaction in aqueous ethanol. The solvent was recycled five times with less than 8% loss in catalytic performance."
    },
    {
        "id": "C013",
        "title": "Supercritical CO2 as an Extraction and Reaction Medium in Flow Chemistry",
        "authors": "Larsen, P. et al.",
        "year": 2022,
        "journal": "Green Chem. Reports",
        "topic": "green_chemistry",
        "abstract": "Supercritical carbon dioxide was used both as reaction solvent and in-line extraction medium for a hydrogenation-extraction tandem flow process. Tunable density of scCO2 allowed switching between homogeneous reaction conditions and biphasic product separation simply by adjusting pressure, eliminating a separate liquid-liquid extraction step and reducing organic solvent consumption by an estimated 95% relative to a conventional batch protocol."
    },
    {
        "id": "C014",
        "title": "Water as the Sole Solvent for Pd-Catalyzed Allylic Substitution",
        "authors": "Mensah, K. et al.",
        "year": 2020,
        "journal": "Green Chem. Reports",
        "topic": "green_chemistry",
        "abstract": "A water-soluble Pd/TPPTS catalyst system enables Tsuji-Trost allylic substitution entirely in water, avoiding organic co-solvents commonly required for substrate solubility. Surfactant-free micellar aggregation of hydrophobic allylic substrates in water was sufficient to achieve turnover frequencies comparable to THF-based systems, and simplified catalyst recovery via simple phase separation after reaction."
    },
    {
        "id": "C015",
        "title": "RAFT Polymerization of Acrylamides for Stimuli-Responsive Hydrogels",
        "authors": "Bergstrom, N. et al.",
        "year": 2021,
        "journal": "Polym. Chem. Frontiers",
        "topic": "polymer_chemistry",
        "abstract": "Reversible addition-fragmentation chain transfer (RAFT) polymerization was used to prepare well-defined poly(N-isopropylacrylamide) block copolymers with narrow dispersity (Mw/Mn less than 1.2). The resulting polymers self-assemble into hydrogels exhibiting a sharp lower critical solution temperature transition near 32 C, tunable by copolymerization with hydrophilic comonomers, with applications in controlled drug release."
    },
    {
        "id": "C016",
        "title": "ATRP-Derived Block Copolymers for Battery Separator Membranes",
        "authors": "Wojcik, M. et al.",
        "year": 2022,
        "journal": "Polym. Chem. Frontiers",
        "topic": "polymer_chemistry",
        "abstract": "Atom transfer radical polymerization was used to synthesize triblock copolymers combining a mechanically robust polystyrene domain with an ion-conducting poly(ethylene oxide) domain. Cast as thin films, the microphase-separated copolymers function as battery separator membranes with ionic conductivities up to 0.4 mS/cm at room temperature, while maintaining sufficient mechanical integrity to suppress lithium dendrite penetration."
    },
    {
        "id": "C017",
        "title": "Ring-Opening Metathesis Polymerization of Norbornene Derivatives",
        "authors": "Delacroix, F. et al.",
        "year": 2020,
        "journal": "Polym. Chem. Frontiers",
        "topic": "polymer_chemistry",
        "abstract": "Grubbs third-generation catalyst was used for living ring-opening metathesis polymerization of functionalized norbornene monomers, giving polymers with predictable molecular weight and dispersity below 1.1. Post-polymerization functionalization via thiol-ene click chemistry installed pendant fluorescent tags without disrupting the polymer backbone, enabling straightforward tracking of the material in downstream processing."
    },
    {
        "id": "C018",
        "title": "Self-Healing Polymer Networks via Dynamic Diels-Alder Crosslinks",
        "authors": "Yamamoto, H. et al.",
        "year": 2023,
        "journal": "Polym. Chem. Frontiers",
        "topic": "polymer_chemistry",
        "abstract": "Thermoreversible Diels-Alder crosslinks between furan and maleimide pendant groups were incorporated into a polyurethane network, granting the material self-healing capability upon mild heating (80-90 C). Mechanical testing showed recovery of over 85% of original tensile strength after three damage-heal cycles, suggesting utility in coatings requiring extended service life."
    },
    {
        "id": "C019",
        "title": "High-Nickel Layered Cathodes for Long-Cycle-Life Lithium-Ion Batteries",
        "authors": "Choudhury, R. et al.",
        "year": 2021,
        "journal": "Electrochem. Energy Mater.",
        "topic": "electrochemistry",
        "abstract": "NMC811 cathode materials were synthesized via co-precipitation and evaluated for cycling stability in lithium-ion full cells. A tungsten-doped surface coating suppressed cation mixing at the particle surface, improving capacity retention from 78% to 92% after 500 cycles at 1C, while maintaining an initial discharge capacity above 200 mAh/g."
    },
    {
        "id": "C020",
        "title": "Solid Polymer Electrolytes for Room-Temperature Sodium-Ion Batteries",
        "authors": "Berg, A. et al.",
        "year": 2022,
        "journal": "Electrochem. Energy Mater.",
        "topic": "electrochemistry",
        "abstract": "A polyethylene oxide-based solid electrolyte doped with sodium bis(trifluoromethanesulfonyl)imide was characterized for use in sodium-ion batteries. Ionic conductivity reached 1.2 x 10^-4 S/cm at 25 C, sufficient for slow-charge applications, and the electrolyte suppressed dendrite formation over 200 plating/stripping cycles in symmetric cells."
    },
    {
        "id": "C021",
        "title": "Redox Flow Batteries Using Water-Soluble Quinone Anolytes",
        "authors": "Petrova, I. et al.",
        "year": 2019,
        "journal": "Electrochem. Energy Mater.",
        "topic": "electrochemistry",
        "abstract": "An anthraquinone disulfonic acid anolyte was paired with a bromide catholyte in an aqueous redox flow battery, achieving a round-trip energy efficiency of 82% at 100 mA/cm2. Degradation studies over 500 cycles identified quinone dimerization as the primary capacity fade mechanism, motivating future substituent modifications to improve stability."
    },
    {
        "id": "C022",
        "title": "Electrocatalytic CO2 Reduction on Copper-Based Nanostructured Electrodes",
        "authors": "Nguyen, T. et al.",
        "year": 2023,
        "journal": "Electrochem. Energy Mater.",
        "topic": "electrochemistry",
        "abstract": "Dendritic copper electrodeposits were evaluated for electrochemical CO2 reduction, showing enhanced selectivity toward C2+ products (ethylene, ethanol) relative to polycrystalline copper foil. Faradaic efficiency for C2+ products reached 58% at -0.9 V vs RHE, attributed to increased local CO concentration within the dendritic nanostructure promoting C-C coupling."
    },
    {
        "id": "C023",
        "title": "Structure-Activity Relationships of Kinase Inhibitor Scaffolds",
        "authors": "Alberti, G. et al.",
        "year": 2021,
        "journal": "Med. Chem. Insights",
        "topic": "medicinal_chemistry",
        "abstract": "A series of pyrazolopyrimidine analogs were synthesized and evaluated against a panel of tyrosine kinases to establish structure-activity relationships. Introduction of a fluorine substituent at the 4-position improved selectivity for the target kinase by 15-fold while reducing off-target activity on a structurally related kinase implicated in cardiotoxicity."
    },
    {
        "id": "C024",
        "title": "Prodrug Strategies for Improving Oral Bioavailability of Poorly Soluble APIs",
        "authors": "Osei, B. et al.",
        "year": 2020,
        "journal": "Med. Chem. Insights",
        "topic": "medicinal_chemistry",
        "abstract": "Phosphate ester prodrugs were designed to improve the aqueous solubility and oral bioavailability of a poorly water-soluble antifungal candidate. In vivo pharmacokinetic studies in rats showed a 4-fold increase in area-under-curve exposure relative to the parent compound, with rapid enzymatic conversion back to active drug in plasma."
    },
    {
        "id": "C025",
        "title": "Fragment-Based Drug Discovery Targeting Protein-Protein Interactions",
        "authors": "Lindqvist, E. et al.",
        "year": 2022,
        "journal": "Med. Chem. Insights",
        "topic": "medicinal_chemistry",
        "abstract": "A fragment library was screened by surface plasmon resonance against a shallow protein-protein interaction interface implicated in inflammatory signaling. Two fragment hits were merged and elaborated through structure-guided synthesis into a lead compound with low micromolar binding affinity, representing a starting point for further optimization of this traditionally difficult target class."
    },
    {
        "id": "C026",
        "title": "Late-Stage C-H Fluorination for Pharmaceutical Analog Synthesis",
        "authors": "Kowalski, D. et al.",
        "year": 2019,
        "journal": "Med. Chem. Insights",
        "topic": "medicinal_chemistry",
        "abstract": "A silver-mediated late-stage C-H fluorination protocol was applied directly to complex drug-like molecules, avoiding the need for de novo synthesis of fluorinated analogs. The method tolerates esters, amides, and heterocycles, enabling rapid generation of a fluorine-scanning analog series for metabolic stability assessment."
    },
    {
        "id": "C027",
        "title": "Two-Dimensional NMR Assignment Strategies for Complex Natural Products",
        "authors": "Tanaka, R. et al.",
        "year": 2020,
        "journal": "Spectrosc. Applied",
        "topic": "spectroscopy",
        "abstract": "A combined HSQC-COSY-HMBC workflow is presented for full structural assignment of a polyketide natural product isolated in submilligram quantity. Careful analysis of long-range HMBC correlations resolved an ambiguity in ring connectivity that could not be settled from 1D NMR data alone, illustrating the value of multi-dimensional NMR for structure elucidation of scarce natural products."
    },
    {
        "id": "C028",
        "title": "In Situ Raman Spectroscopy for Monitoring Catalytic Reaction Progress",
        "authors": "Fischer, J. et al.",
        "year": 2021,
        "journal": "Spectrosc. Applied",
        "topic": "spectroscopy",
        "abstract": "A fiber-optic Raman probe was integrated into a batch reactor to monitor a heterogeneously catalyzed hydrogenation in real time. Characteristic vibrational bands for the alkene starting material and saturated product were tracked throughout the reaction, allowing determination of reaction kinetics without the need for offline sampling and chromatographic analysis."
    },
    {
        "id": "C029",
        "title": "Infrared Spectroscopic Signatures of Nitro Group Vibrations in Energetic Materials",
        "authors": "Volkov, S. et al.",
        "year": 2022,
        "journal": "Spectrosc. Applied",
        "topic": "spectroscopy",
        "abstract": "FTIR spectra of a series of nitroaromatic and nitramine compounds were analyzed to catalog characteristic asymmetric and symmetric NO2 stretching frequencies. Systematic shifts in stretching frequency correlated with electron density at the nitro-bearing carbon, providing a spectroscopic handle for rapid, non-destructive classification of unknown nitro-containing compounds in field settings."
    },
    {
        "id": "C030",
        "title": "Chemometric Analysis of UV-Vis Spectra for Reaction Endpoint Detection",
        "authors": "Moreau, C. et al.",
        "year": 2023,
        "journal": "Spectrosc. Applied",
        "topic": "spectroscopy",
        "abstract": "Principal component analysis was applied to time-resolved UV-Vis spectra collected during a multistep one-pot synthesis, enabling automated detection of reaction endpoints without manual spectral interpretation. The chemometric model correctly flagged completion within 2 minutes of the endpoint determined by offline HPLC in 18 of 20 test runs."
    },
    {
        "id": "C031",
        "title": "DFT Benchmarking of Exchange-Correlation Functionals for Transition Metal Catalysis",
        "authors": "Andersen, K. et al.",
        "year": 2021,
        "journal": "Comput. Chem. Studies",
        "topic": "computational_chemistry",
        "abstract": "A benchmark study compares B3LYP, M06, and omega-B97X-D functionals against high-level CCSD(T) reference energies for a set of palladium oxidative addition transition states relevant to cross-coupling catalysis. omega-B97X-D gave the closest agreement with reference barriers, while B3LYP systematically underestimated activation energies by 3-5 kcal/mol, a finding relevant to computational screening of new cross-coupling catalyst designs."
    },
    {
        "id": "C032",
        "title": "Machine-Learned Interatomic Potentials for Accelerated Molecular Dynamics",
        "authors": "Zhou, W. et al.",
        "year": 2022,
        "journal": "Comput. Chem. Studies",
        "topic": "computational_chemistry",
        "abstract": "A neural network interatomic potential trained on DFT single-point energies was used to accelerate molecular dynamics simulations of a solid-electrolyte interphase model by three orders of magnitude relative to ab initio MD, while reproducing radial distribution functions within 5% of the reference DFT-MD trajectory."
    },
    {
        "id": "C033",
        "title": "QSPR Modeling of Detonation Velocity from Molecular Descriptors",
        "authors": "Hassan, N. et al.",
        "year": 2020,
        "journal": "Comput. Chem. Studies",
        "topic": "computational_chemistry",
        "abstract": "A group-additivity quantitative structure-property relationship model was developed to predict detonation velocity of energetic compounds directly from molecular substructure counts, without requiring prior identification of the specific compound. The model achieved a mean absolute error of under 3% across a validation set spanning nitroaromatic, nitramine, and nitrate ester compound classes, and was shown to generalize reasonably to novel, previously uncharacterized candidate molecules."
    },
    {
        "id": "C034",
        "title": "Transfer Learning for Reaction Yield Prediction Across Chemical Reaction Classes",
        "authors": "Ferreira, L. et al.",
        "year": 2023,
        "journal": "Comput. Chem. Studies",
        "topic": "computational_chemistry",
        "abstract": "A transformer-based model pretrained on a large corpus of high-throughput experimentation data was fine-tuned to predict yields for a new class of reactions with only a few hundred labeled examples. Transfer learning improved prediction R2 from 0.41 (trained from scratch) to 0.77 (fine-tuned), demonstrating the value of pretraining for low-data reaction optimization campaigns."
    },
    {
        "id": "C035",
        "title": "Mesoporous Silica-Supported Palladium Nanoparticles for Heterogeneous Catalysis",
        "authors": "Kim, S. et al.",
        "year": 2021,
        "journal": "Nano Catal. Today",
        "topic": "nanomaterials",
        "abstract": "Palladium nanoparticles supported on SBA-15 mesoporous silica were prepared by wet impregnation and evaluated for hydrogenation of nitroarenes. The confined mesopore environment limited nanoparticle sintering during repeated use, maintaining greater than 90% conversion over ten reaction cycles, compared to a 40% drop in activity for unsupported palladium black over the same number of cycles."
    },
    {
        "id": "C036",
        "title": "Gold Nanoparticle Catalysts for Aerobic Alcohol Oxidation",
        "authors": "Rossi, F. et al.",
        "year": 2020,
        "journal": "Nano Catal. Today",
        "topic": "nanomaterials",
        "abstract": "Citrate-stabilized gold nanoparticles of controlled size (3-8 nm) were tested for aerobic oxidation of benzylic alcohols to aldehydes using molecular oxygen as the terminal oxidant. Catalytic activity scaled inversely with particle size, consistent with a surface-area-dependent mechanism, and the smallest nanoparticles achieved full conversion within 4 hours at 80 C without over-oxidation to the carboxylic acid."
    },
    {
        "id": "C037",
        "title": "Metal-Organic Framework Catalysts for Selective CO2 Cycloaddition",
        "authors": "Aboud, Y. et al.",
        "year": 2022,
        "journal": "Nano Catal. Today",
        "topic": "nanomaterials",
        "abstract": "A zinc-based metal-organic framework bearing accessible Lewis acidic sites was evaluated as a heterogeneous catalyst for cycloaddition of CO2 with epoxides to form cyclic carbonates. The framework maintained catalytic activity across five recycling runs and outperformed a homogeneous zinc salt analog under identical conditions, attributed to the confined pore environment concentrating both reactants near the active site."
    },
    {
        "id": "C038",
        "title": "Discovery of a Tandem C-H Borylation/Suzuki Relay: The Kessler-Onyema Coupling",
        "authors": "Kessler, F. and Onyema, U.",
        "year": 2026,
        "journal": "Frontiers in Cross-Coupling Chemistry",
        "topic": "fictional_reaction",
        "abstract": "We report a previously undescribed tandem process, herein termed the Kessler-Onyema coupling, in which an Ir-catalyzed C-H borylation is directly relayed into a Suzuki-Miyaura coupling within a single reaction vessel, without isolation of the intermediate boronate ester. The two catalytic cycles are compatibilized using a mixed dioxane/water/tert-amyl alcohol solvent system, which supports both the anhydrous borylation step and the subsequent aqueous-base-mediated coupling step. Across 19 heteroarene substrates, one-pot yields of the biaryl products averaged 71%, outperforming the corresponding two-step, two-solvent sequence (average 52% over two steps) and eliminating a chromatographic purification of the boronate intermediate. Catalyst loadings of 2 mol% Ir and 3 mol% Pd were required; lower Pd loadings led to incomplete consumption of the in-situ-generated boronate before protodeboronation became competitive. This tandem strategy is expected to streamline the synthesis of unsymmetrical heterobiaryl fragments common in agrochemical and pharmaceutical discovery programs."
    },
    {
        "id": "C039",
        "title": "Organocatalytic Enantioselective Michael Additions Using Cinchona Alkaloids",
        "authors": "Bianchi, E. et al.",
        "year": 2019,
        "journal": "Adv. Catal. Lett.",
        "topic": "misc_catalysis",
        "abstract": "A cinchona-alkaloid-derived thiourea catalyst promotes enantioselective Michael addition of malonate nucleophiles to nitroolefins, giving products in up to 96% ee. The reaction proceeds in toluene at -20 C, and catalyst loading could be reduced to 2 mol% without loss of enantioselectivity, making the protocol practical for gram-scale synthesis of chiral building blocks."
    },
    {
        "id": "C040",
        "title": "Directed C-H Activation for Site-Selective Arene Functionalization",
        "authors": "Meunier, P. et al.",
        "year": 2021,
        "journal": "Adv. Catal. Lett.",
        "topic": "misc_catalysis",
        "abstract": "A removable pyridine-based directing group enables palladium-catalyzed ortho C-H functionalization of arylacetic acid derivatives with high site selectivity. The directing group is installed and removed in one step each, and the overall sequence was applied to introduce iodide, acetoxy, and alkenyl substituents at the previously inaccessible ortho position of 12 substrates."
    }
]

In [ ]:
# ---- Option B: full 500-abstract PubMed corpus ----
# Set this to the local path or raw-GitHub URL of the CSV built by data_prep_pubmed.py
# (columns: id, title, authors, year, journal, topic, abstract)
FULL_CORPUS_PATH = None  # e.g. "https://raw.githubusercontent.com/<you>/ai4chemistry-bootcamp/main/data/chemistry_abstracts.csv"

if FULL_CORPUS_PATH:
    corpus = pd.read_csv(FULL_CORPUS_PATH).to_dict("records")
    print(f"Loaded {len(corpus)} abstracts from {FULL_CORPUS_PATH}")
else:
    corpus = DEMO_CORPUS
    print(f"Loaded {len(corpus)} abstracts from the built-in demo corpus")

## Part 1 — Chunk and Embed

Each abstract needs to be split into overlapping chunks before embedding, so that (a) very long abstracts or full papers don't get truncated by the embedding model's context window, and (b) retrieval can return a focused passage instead of an entire paper.

**Note on chunk size:** the assignment spec calls for 200-token chunks with 20-token overlap — sized for *full paper text* or longer abstracts. Our demo abstracts are short (40–140 words), so we use smaller `CHUNK_SIZE` / `CHUNK_OVERLAP` values here to actually see multi-chunk behavior in a small corpus. Switch to `CHUNK_SIZE = 200, CHUNK_OVERLAP = 20` once you move to the full PubMed corpus.

In [ ]:
CHUNK_SIZE = 60      # words; use 200 for the full PubMed corpus per the assignment spec
CHUNK_OVERLAP = 15   # words; use 20 for the full corpus

### 🧩 Exercise 1 — `chunk_text()`
Implement word-based chunking with overlap. Given a string, split it into a list of chunks of `chunk_size` words each, where consecutive chunks share `overlap` words.

In [ ]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """
    Split `text` into overlapping chunks of `chunk_size` words, where
    consecutive chunks share `overlap` words.

    Parameters
    ----------
    text : str
    chunk_size : int
    overlap : int

    Returns
    -------
    list[str]
    """
    ### YOUR CODE HERE ###
    pass


# quick sanity check
_test_chunks = chunk_text("word " * 130, chunk_size=60, overlap=15)
print(f"{len(_test_chunks)} chunks produced from 130 words (expect 3 with chunk_size=60, overlap=15)")

### 🧩 Exercise 1 (continued) — Embed chunks and build the FAISS index

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

chunk_records = []  # each: {"chunk_id", "paper_id", "title", "topic", "text"}
for paper in corpus:
    for i, ch in enumerate(chunk_text(paper["abstract"])):
        chunk_records.append({
            "chunk_id": f"{paper['id']}_chunk{i}",
            "paper_id": paper["id"],
            "title": paper["title"],
            "topic": paper.get("topic", ""),
            "text": ch,
        })

print(f"{len(corpus)} abstracts -> {len(chunk_records)} chunks")

### YOUR CODE HERE ###
# 1. Embed every chunk's text with `embedder.encode(...)`, as a float32 numpy array
#    -> chunk_embeddings
# 2. Create a FAISS flat L2 index (faiss.IndexFlatL2) with the right dimension
#    -> index
# 3. Add chunk_embeddings to the index
chunk_embeddings = None
index = None

print(f"FAISS index built: {index.ntotal if index else 0} vectors")

## Part 2 — Retrieval

### 🧩 Exercise 2 — `retrieve(query, k=5)`
Embed the query with the *same* model used for the chunks, search the FAISS index for the `k` nearest neighbours, and return them together with their source metadata (paper title, chunk id, distance).

In [ ]:
def retrieve(query, k=5):
    """
    Embed `query`, search the FAISS index, and return the top-k chunks
    with their source metadata.

    Returns
    -------
    list[dict] with keys: rank, distance, chunk_id, paper_id, title, text
    """
    ### YOUR CODE HERE ###
    pass

### 🗣️ Question 1 (Warm-up)
Run `retrieve()` below on a query clearly *inside* the corpus (e.g. a battery or polymer question) and one clearly *outside* it (e.g. "What is the boiling point of ethanol?").

- What does the FAISS index return for the out-of-corpus query? Does it refuse, or does it just return the *least bad* chunks it has?
- What does that tell you about the difference between a retriever "knowing nothing" and an LLM "knowing nothing"?

_Your answer:_

In [ ]:
for q in ["How does particle size affect gold nanoparticle catalysis?",
          "What is the boiling point of ethanol?"]:
    print(f"Query: {q}")
    for r in retrieve(q, k=3):
        print(f"  [{r['rank']}] dist={r['distance']:.3f}  {r['title']}")
    print()

### 🗣️ Question 2 (Easy)
For the query **"What solvents are used in Suzuki coupling?"**, look at the top-3 retrieved chunks printed below.

- Do all three come from different papers, or does one paper dominate?
- Do they actually answer the question, or just mention "Suzuki coupling" without discussing solvents?

In [ ]:
suzuki_query = "What solvents are used in Suzuki coupling?"
for r in retrieve(suzuki_query, k=3):
    print(f"[{r['rank']}] {r['title']}  (dist={r['distance']:.3f})")
    print(f"    {r['text']}\n")

_Your answer:_

## Part 3 — Build the RAG Chain

### 🧩 Exercise 3 — `build_prompt(query, retrieved_chunks)`
Build a prompt that: numbers each retrieved chunk, includes its source title, instructs the model to answer only from the provided context (and to say so if it can't), and to cite chunk numbers inline (e.g. `[2]`). Return the full prompt as a single string.

In [ ]:
def build_prompt(query, retrieved_chunks):
    """
    Construct a RAG prompt: numbered context passages + instructions + the question.
    """
    ### YOUR CODE HERE ###
    pass

The LLM call itself is provided below, with a free local fallback if you don't have an OpenAI API key handy — no exercise here, just infrastructure.

In [ ]:
import os

# This bootcamp session runs entirely on the free local fallback model -
# no OpenAI API key needed. (If you ever want to swap in OpenAI later,
# just set OPENAI_API_KEY and set USE_OPENAI = True below.)
USE_OPENAI = False

# Using AutoTokenizer/AutoModelForSeq2SeqLM directly (rather than the
# transformers.pipeline() helper) since some transformers builds don't
# register the "text2text-generation" pipeline task - this call works
# the same way across versions.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

_local_model_name = "google/flan-t5-base"
_local_tokenizer = AutoTokenizer.from_pretrained(_local_model_name)
_local_model = AutoModelForSeq2SeqLM.from_pretrained(_local_model_name)

def generate_answer(prompt, model=None, max_tokens=256):
    inputs = _local_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    output_ids = _local_model.generate(**inputs, max_new_tokens=max_tokens)
    return _local_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

print("Using local flan-t5-base fallback (no API key required)")


In [ ]:
def rag_answer(query, k=5, verbose=True):
    retrieved = retrieve(query, k=k)
    prompt = build_prompt(query, retrieved)
    answer = generate_answer(prompt)
    if verbose:
        print(f"Q: {query}\n")
        print(f"A: {answer}\n")
        print("Sources:")
        for r in retrieved:
            print(f"  [{r['rank']}] {r['title']}")
    return {"query": query, "answer": answer, "retrieved": retrieved, "prompt": prompt}


_ = rag_answer(suzuki_query, k=3)

## Part 4 — RAG vs. Bare LLM

In [ ]:
def bare_llm_answer(query):
    prompt = f"Answer the following chemistry question as accurately as you can:\n\n{query}"
    return generate_answer(prompt)

### 🗣️ Question 3 (Medium)
The query below asks about the (deliberately fictional) **Kessler-Onyema coupling** — a reaction that appears nowhere in any LLM's training data, but *does* appear in our corpus (`C038`).

- Does the bare LLM admit it doesn't know, or does it confidently invent a plausible-sounding answer? Look closely at specific details (solvents, catalyst loadings, yields) — are they fabricated?
- Does the RAG answer correctly ground itself in the retrieved abstract?
- Now try this with a *real* recent or obscure reaction/topic of your choice instead. Does the bare LLM still hallucinate, or does it correctly say "I'm not sure"? What does that suggest about how (and whether) LLMs signal their own uncertainty?

In [ ]:
recent_reaction_query = "Describe the Kessler-Onyema coupling and what solvent system it uses."

print("=== Bare LLM (no retrieval) ===")
print(bare_llm_answer(recent_reaction_query))

print("\n=== RAG (with retrieval) ===")
_ = rag_answer(recent_reaction_query, k=3)

_Your answer:_

## Part 5 — Evaluate: Faithfulness & Relevance

Two different questions matter for a RAG answer, and they can disagree with each other:

- **Faithfulness** — is the answer actually supported by the retrieved context, or is the model leaning on its own (possibly wrong, possibly outdated) pretrained knowledge?
- **Relevance** — does the answer address what was actually asked, regardless of how well-supported it is?

A faithful-but-irrelevant answer and a relevant-but-unfaithful answer are both failures, just different ones.

### 🧩 Exercise 4a — `faithfulness_score()`
Implement faithfulness as BERTScore **recall** between the generated answer and the concatenated retrieved context: how much of the context's content shows up in the answer, and vice versa in spirit — practically, how well-grounded the answer is in what was retrieved.

In [ ]:
from bert_score import score as bertscore

def faithfulness_score(answer, retrieved_chunks):
    """
    Approximate faithfulness: BERTScore recall of the answer against the
    concatenated retrieved context.
    """
    ### YOUR CODE HERE ###
    pass

### 🧩 Exercise 4b — `relevance_score()`
Implement relevance as the cosine similarity between the sentence-transformer embeddings of the question and the answer.

In [ ]:
def relevance_score(answer, question):
    """
    Approximate relevance: cosine similarity between the question and
    answer embeddings.
    """
    ### YOUR CODE HERE ###
    pass

In [ ]:
EVAL_QUESTIONS = [
    "What solvents are used in Suzuki coupling?",
    "How does ligand-free Suzuki coupling in ethanol/water compare to using an XPhos precatalyst?",
    "What is the Kessler-Onyema coupling and what solvent system does it use?",
    "What functional groups are tolerated in late-stage C-H fluorination?",
    "How is a deep eutectic solvent used in aldol condensations?",
    "What ionic conductivity is reported for the sodium-ion solid polymer electrolyte in the corpus?",
    "Which DFT functional gives the best agreement with CCSD(T) for palladium oxidative addition barriers?",
    "How does particle size affect gold-nanoparticle-catalyzed alcohol oxidation?",
    "What is the main degradation pathway in the quinone-based redox flow battery?",
    "How is RAFT polymerization used to make stimuli-responsive hydrogels?",
]

rows = []
for q in EVAL_QUESTIONS:
    t0 = time.time()
    result = rag_answer(q, k=5, verbose=False)
    latency = time.time() - t0
    rows.append({
        "question": q,
        "latency_s": latency,
        "faithfulness": faithfulness_score(result["answer"], result["retrieved"]),
        "relevance": relevance_score(result["answer"], q),
    })

eval_df = pd.DataFrame(rows)
eval_df

## Part 6 — Does More Retrieval Help? (k = 3 → 10)

### 🗣️ Question 4 (Medium)
Run the sweep below over `k = [3, 5, 7, 10]` on the 10 evaluation questions.

- Does faithfulness keep improving as `k` grows, or does it plateau (or even drop)?
- How does latency scale with `k`? Where's the point of diminishing returns for *this* corpus?
- Would you expect that point to shift with the full 500-abstract corpus, or with full paper text instead of abstracts? Why?

In [ ]:
k_values = [3, 5, 7, 10]
k_results = []

for k in k_values:
    latencies, faiths = [], []
    for q in EVAL_QUESTIONS:
        t0 = time.time()
        result = rag_answer(q, k=k, verbose=False)
        latencies.append(time.time() - t0)
        faiths.append(faithfulness_score(result["answer"], result["retrieved"]))
    k_results.append({
        "k": k,
        "avg_latency_s": sum(latencies) / len(latencies),
        "avg_faithfulness": sum(faiths) / len(faiths),
    })

k_df = pd.DataFrame(k_results)
print(k_df)

fig, ax1 = plt.subplots(figsize=(6, 4))
ax1.plot(k_df["k"], k_df["avg_faithfulness"], marker="o", color="tab:blue", label="Faithfulness")
ax1.set_xlabel("k (retrieved chunks)")
ax1.set_ylabel("Avg. faithfulness", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(k_df["k"], k_df["avg_latency_s"], marker="s", color="tab:red", label="Latency (s)")
ax2.set_ylabel("Avg. latency (s)", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")

plt.title("Effect of k on answer quality vs. latency")
fig.tight_layout()
plt.show()

_Your answer:_

## Part 7 — Challenge: Cross-Encoder Re-ranking

FAISS retrieval uses a fast **bi-encoder**: the query and each chunk are embedded *independently*, so retrieval is just a nearest-neighbour search. A **cross-encoder** instead scores a `(query, chunk)` pair jointly, letting it model interactions between the two texts — more accurate, but far too slow to run against the whole corpus. The standard pattern is to use the cheap bi-encoder to get a shortlist, then the expensive cross-encoder to re-rank just that shortlist.

### 🧩 Exercise 5 (Challenge) — `rerank(query, candidates, top_n=3)`
Use `cross-encoder/ms-marco-MiniLM-L-6-v2` to re-score the FAISS candidates and return the best `top_n`.

In [ ]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, candidates, top_n=3):
    """
    Re-score `candidates` (a list of retrieve()-style dicts) against `query`
    using the cross-encoder, and return the best `top_n`, sorted descending
    by rerank score. Also store the score on each returned dict as
    "rerank_score".
    """
    ### YOUR CODE HERE ###
    pass

In [ ]:
def rag_answer_reranked(query, k=10, top_n=3, verbose=True):
    candidates = retrieve(query, k=k)
    top_chunks = rerank(query, candidates, top_n=top_n)
    prompt = build_prompt(query, top_chunks)
    answer = generate_answer(prompt)
    if verbose:
        print(f"Q: {query}\n\nA: {answer}\n")
        print("Sources (after re-ranking):")
        for c in top_chunks:
            print(f"  {c['title']} (rerank_score={c['rerank_score']:.3f})")
    return {"query": query, "answer": answer, "retrieved": top_chunks}


rows = []
for q in EVAL_QUESTIONS:
    base = rag_answer(q, k=3, verbose=False)
    reranked = rag_answer_reranked(q, k=10, top_n=3, verbose=False)
    rows.append({
        "question": q,
        "faithfulness_no_rerank": faithfulness_score(base["answer"], base["retrieved"]),
        "faithfulness_reranked": faithfulness_score(reranked["answer"], reranked["retrieved"]),
    })

rerank_df = pd.DataFrame(rows)
rerank_df["improvement"] = rerank_df["faithfulness_reranked"] - rerank_df["faithfulness_no_rerank"]
rerank_df

### 🗣️ Question 5 (Challenge)
Look at the `improvement` column above.

- Does re-ranking improve faithfulness on average? Is the improvement consistent across questions, or concentrated in a few?
- Re-ranking costs an extra model call per candidate (10 cross-encoder calls vs. one FAISS search). Under what circumstances would that added latency/cost be worth it in a production chemistry-literature assistant?

_Your answer:_

## Wrap-up

You've now built a complete RAG pipeline — chunking, embedding, FAISS retrieval, prompt construction, generation, evaluation, and re-ranking — and seen concretely where it helps (grounding answers in real sources, reducing hallucination) and where it has limits (short abstracts don't chunk much, more retrieved context isn't always better, faithfulness and relevance are separate axes that can disagree).

**Ideas to take further:**
- Swap in the full 500-abstract PubMed corpus (`data_prep_pubmed.py`) and repeat Question 4 — does the diminishing-returns point shift?
- Try a hybrid retriever (BM25 + dense embeddings) and see if faithfulness improves further.
- Add an LLM-as-judge relevance metric and compare it against the embedding-cosine proxy used here — where do they disagree?